# 01 Data
Load raw BLUME data, check structure, coverage and drops

In [1]:
import json
from pathlib import Path

import pandas as pd

# Project root is one level up (notebook runs in notebooks/); raw/ holds API data, never modified
ROOT = Path.cwd().parent
RAW = ROOT / "raw"

print(RAW)
print(RAW.exists())

/Users/creative/Air_Quality_Project/raw
True


In [2]:
# Raw files, sorted so the order is the same on every run
files = sorted(RAW.glob("*.json"))

print(len(files))
for f in files[:5]:
    print(f.name)


324
2025-01_mc010_no2.json
2025-01_mc010_pm2.json
2025-01_mc027_no2.json
2025-01_mc032_no2.json
2025-01_mc032_pm2.json


In [3]:
# Open one raw file to see its structure before loading all
data = json.loads(files[0].read_text())

print(files[0].name)
print(type(data))

2025-01_mc010_no2.json
<class 'list'>


In [4]:
# Inspect one file: size, first and last entry (shows order and time span)
print(len(data))
print(data[0])
print(data[-1])

744
{'datetime': '2025-02-01T00:00:00+01:00', 'station': 'mc010', 'core': 'no2', 'component': 'no2_1h', 'period': '1h', 'value': 24.0}
{'datetime': '2025-01-01T01:00:00+01:00', 'station': 'mc010', 'core': 'no2', 'component': 'no2_1h', 'period': '1h', 'value': 32.0}


**Findings**
- Expected hours per month in 2025: Jan (744), Feb (672), Mar (743, daylight saving starts), Apr (720), May (744), Jun (720), Jul (744), Aug (744), Sep (720), Oct (745, daylight saving ends), Nov (720), Dec (744). Year (8,760).
- January, mc010, NO₂: 744 hours. Complete.

In [5]:
# Expected hours per month in 2025, Berlin time
# March 743 and October 745: clock change to/from summer time
EXPECTED = {
    "01": 744, "02": 672, "03": 743, "04": 720, "05": 744, "06": 720,
    "07": 744, "08": 744, "09": 720, "10": 745, "11": 720, "12": 744,
}

# Print only files whose entry count differs from expected
for f in files:
    month = f.name[5:7]                                  # "2025-01_mc010_no2.json" → "01"
    n = len(json.loads(f.read_text()))
    if n != EXPECTED[month]:
        print(f.name, n, "expected", EXPECTED[month])

2025-01_mc174_no2.json 743 expected 744
2025-10_mc010_no2.json 744 expected 745
2025-10_mc010_pm2.json 744 expected 745
2025-10_mc027_no2.json 744 expected 745
2025-10_mc032_no2.json 744 expected 745
2025-10_mc032_pm2.json 744 expected 745
2025-10_mc042_no2.json 744 expected 745
2025-10_mc042_pm2.json 744 expected 745
2025-10_mc077_no2.json 744 expected 745
2025-10_mc077_pm2.json 744 expected 745
2025-10_mc085_no2.json 744 expected 745
2025-10_mc085_pm2.json 744 expected 745
2025-10_mc117_no2.json 744 expected 745
2025-10_mc117_pm2.json 744 expected 745
2025-10_mc124_no2.json 744 expected 745
2025-10_mc124_pm2.json 744 expected 745
2025-10_mc144_no2.json 744 expected 745
2025-10_mc144_pm2.json 744 expected 745
2025-10_mc145_no2.json 744 expected 745
2025-10_mc171_no2.json 744 expected 745
2025-10_mc171_pm2.json 744 expected 745
2025-10_mc174_no2.json 744 expected 745
2025-10_mc174_pm2.json 744 expected 745
2025-10_mc190_no2.json 744 expected 745
2025-10_mc190_pm2.json 744 expected 745


In [6]:
# October night of the clock change: check how the doubled hour is stored
oct_file = RAW / "2025-10_mc010_no2.json"
oct_data = json.loads(oct_file.read_text())

for entry in oct_data:
    if entry["datetime"].startswith("2025-10-26T0"):    # 26 Oct, 00:00–09:00
        print(entry["datetime"], entry["value"])

2025-10-26T09:00:00+01:00 9.0
2025-10-26T08:00:00+01:00 8.0
2025-10-26T07:00:00+01:00 7.0
2025-10-26T06:00:00+01:00 5.0
2025-10-26T05:00:00+01:00 2.0
2025-10-26T04:00:00+01:00 2.0
2025-10-26T03:00:00+01:00 2.0
2025-10-26T02:00:00+02:00 None
2025-10-26T01:00:00+02:00 6.0
2025-10-26T00:00:00+02:00 8.0


**Findings**
- October: the API skips the second 02:00 on 26 Oct (clock change). One hour missing in every file.
- Some entries have `None` as value: the hour exists, but nothing was measured. Entry counts alone don't show completeness.
- Timestamps include the UTC offset (+01:00 / +02:00), so converting to UTC gives a clean, gap-free timeline.

In [7]:
# missing values (None) per file
for f in files:
    data = json.loads(f.read_text())                            # entries in file
    n_none = sum(1 for entry in data if entry["value"] is None) # count entries with value None
    if n_none > 0:
        print(f.name, n_none)                                   # files with missing values only

2025-01_mc010_no2.json 2
2025-01_mc027_no2.json 1
2025-01_mc032_no2.json 1
2025-01_mc032_pm2.json 2
2025-01_mc042_no2.json 3
2025-01_mc077_no2.json 2
2025-01_mc085_no2.json 25
2025-01_mc085_pm2.json 24
2025-01_mc117_no2.json 2
2025-01_mc117_pm2.json 1
2025-01_mc124_no2.json 2
2025-01_mc124_pm2.json 1
2025-01_mc144_no2.json 3
2025-01_mc145_no2.json 2
2025-01_mc171_no2.json 3
2025-01_mc174_no2.json 3
2025-01_mc174_pm2.json 1
2025-01_mc190_no2.json 1
2025-01_mc221_no2.json 3
2025-01_mc282_no2.json 1
2025-02_mc010_no2.json 1
2025-02_mc144_no2.json 1
2025-02_mc144_pm2.json 1
2025-02_mc145_no2.json 1
2025-02_mc171_pm2.json 2
2025-02_mc282_no2.json 4
2025-03_mc010_no2.json 2
2025-03_mc010_pm2.json 2
2025-03_mc027_no2.json 2
2025-03_mc032_no2.json 3
2025-03_mc032_pm2.json 2
2025-03_mc042_no2.json 2
2025-03_mc042_pm2.json 1
2025-03_mc077_no2.json 2
2025-03_mc077_pm2.json 2
2025-03_mc085_no2.json 4
2025-03_mc085_pm2.json 56
2025-03_mc117_no2.json 2
2025-03_mc117_pm2.json 2
2025-03_mc124_no2.json

In [8]:
# Combine all raw monthly files into one table
rows = []
for f in files:
    rows.extend(json.loads(f.read_text()))

df = pd.DataFrame(rows)
df["datetime"] = pd.to_datetime(df["datetime"], utc=True)   # files mix +01:00 and +02:00 offsets

print(df.shape)
df.head()

(236492, 6)


,datetime,station,core,component,period,value
0,2025-01-31 23:00:00+00:00,mc010,no2,no2_1h,1h,24.0
1,2025-01-31 22:00:00+00:00,mc010,no2,no2_1h,1h,23.0
2,2025-01-31 21:00:00+00:00,mc010,no2,no2_1h,1h,30.0
3,2025-01-31 20:00:00+00:00,mc010,no2,no2_1h,1h,28.0
4,2025-01-31 19:00:00+00:00,mc010,no2,no2_1h,1h,23.0


In [9]:
# Timestamps are UTC after loading; Berlin time needed for hour of day and season
df["datetime"] = df["datetime"].dt.tz_convert("Europe/Berlin")

# Timestamps mark end of hour (01:00 = 00:00–01:00); shift to start so hour 7 = 07:00–08:00
df["datetime"] = df["datetime"] - pd.Timedelta(hours=1)

print("first:", df["datetime"].min())
print("last: ", df["datetime"].max())

first: 2025-01-01 00:00:00+01:00
last:  2025-12-31 23:00:00+01:00


In [10]:
# Month files may overlap at the edges; each station, pollutant and hour should appear once
dupes = df.duplicated(subset=["station", "core", "datetime"])
print("duplicates:", dupes.sum())

duplicates: 0


In [11]:
# Drop incomplete days; transform() gives each row its day's valid-hour count
n_valid = df.groupby(["station", "core", df["datetime"].dt.date])["value"].transform("count")

before = len(df)
df = df[n_valid >= 18]
print("rows removed:", before - len(df))
print(df.shape)

rows removed: 312
(236180, 6)


In [12]:
# Negative values are physically impossible; very high ones need a closer look
print("negative:", (df["value"] < 0).sum())
df.groupby("core")["value"].describe()

negative: 0


,count,mean,std,min,25%,50%,75%,max
core,,,,,,,,
no2,130974.0,16.056446,13.075216,2.0,6.0,12.0,22.0,121.0
pm2,104765.0,11.539398,8.419273,1.0,6.0,9.0,14.0,276.0


## Outliers

In [13]:
# Top 10 PM2.5 hours: fireworks or sensor error? Real events hit several stations at once
df[df["core"] == "pm2"].nlargest(10, "value")[["datetime", "station", "value"]]

,datetime,station,value
12647,2025-01-01 00:00:00+01:00,mc144,276.0
16366,2025-01-01 00:00:00+01:00,mc174,250.0
11159,2025-01-01 00:00:00+01:00,mc124,232.0
19342,2025-01-01 00:00:00+01:00,mc221,191.0
14879,2025-01-01 00:00:00+01:00,mc171,162.0
9671,2025-01-01 00:00:00+01:00,mc117,140.0
226820,2025-12-31 23:00:00+01:00,mc124,122.0
232028,2025-12-31 23:00:00+01:00,mc174,112.0
16365,2025-01-01 01:00:00+01:00,mc174,109.0
1487,2025-01-01 00:00:00+01:00,mc010,103.0


In [14]:
# New Year's Eve 18:00 to 1 Jan 06:00, both year ends; kept in data, excluded on demand
dt = df["datetime"]
df["fireworks"] = (
    ((dt.dt.month == 12) & (dt.dt.day == 31) & (dt.dt.hour >= 18))
    | ((dt.dt.month == 1) & (dt.dt.day == 1) & (dt.dt.hour < 6))
)
print("flagged rows:", df["fireworks"].sum())

flagged rows: 323


In [15]:
# Top 10 NO₂ hours: one station alone on a random date could be a sensor error
df[df["core"] == "no2"].nlargest(10, "value")[["datetime", "station", "value"]]

,datetime,station,value
56642,2025-03-07 19:00:00+01:00,mc221,121.0
49956,2025-03-07 18:00:00+01:00,mc144,119.0
49955,2025-03-07 19:00:00+01:00,mc144,114.0
53670,2025-03-07 19:00:00+01:00,mc174,114.0
53358,2025-03-20 19:00:00+01:00,mc174,113.0
56643,2025-03-07 18:00:00+01:00,mc221,112.0
56330,2025-03-20 19:00:00+01:00,mc221,111.0
53357,2025-03-20 20:00:00+01:00,mc174,109.0
56570,2025-03-10 19:00:00+01:00,mc221,107.0
38496,2025-03-20 21:00:00+01:00,mc010,106.0


## Coverage

In [16]:
# Share of the year's 8,760 hours with a valid value, after dropping incomplete days
cov = df.groupby(["station", "core"])["value"].count().rename("valid_hours").to_frame()
cov["pct"] = (cov["valid_hours"] / 8760 * 100).round(1)
cov.sort_values("pct")

,,valid_hours,pct
station,core,,
mc085,pm2,8625,98.5
mc221,pm2,8658,98.8
mc085,no2,8688,99.2
mc032,no2,8688,99.2
mc042,no2,8729,99.6
mc124,no2,8738,99.7
mc145,no2,8738,99.7
mc190,no2,8738,99.7
mc174,no2,8737,99.7


## Findings
- One file per month × station × pollutant. PM2.5 is named `pm2`.
- Timestamps include UTC offset. Newest first.
- All months complete, except mc174 NO₂ Jan (−1) and all October files (−1, clock change).
- Most files miss 1–6 values (`None`). Longer gaps: mc085 Jan, mc085 Mar, mc221 Apr, mc221 Jul, mc032 Dec.
- No duplicate hours (station × pollutant × hour).
- 13 station-days (312 rows) dropped: < 18 valid hours.
- No negative values. Max NO₂ 121, PM2.5 276 µg/m³. Values are whole numbers.
- Top 10 NO₂ hours: evenings 18–21 h in March (7, 10, 20 March), several stations at once points to real event. Peak hours in March, not Dec–Feb (check in 03-season, H2).
- Coverage after drops: 98.5–99.9 % valid hours in all 27 series (15 NO₂, 12 PM2.5), well above the 90 % EU minimum. Lowest: mc085 and mc221 PM2.5.
- PM2.5 is measured hourly at 12 of 15 stations; mc027, mc145 and mc282 measure NO₂ only.

## Decisions
- Missing values are not filled.
- Hours vs. 24-hour levels: indicator only.
- Days vs. 24-hour levels: only with ≥ 18 valid hours.
- Timestamps mark end of hour; shifted −1 h to hour start.
- Fireworks hours (31 Dec 18:00 – 1 Jan 06:00) kept as real exposure; flagged in `fireworks` to check seasonal means without them.